In [ ]:
import cv2
import numpy as np
import os
import mediapipe as mp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.utils.class_weight import compute_class_weight
import joblib
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, BatchNormalization, Dropout,
    Dense, LayerNormalization, Conv1D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEQUENCE_LENGTH  = 30
DATASET_PATH     = '/kaggle/input/datasets/jennifercheriyan/isl-gloss-include'
WORKING_DIR      = '/kaggle/working'

mp_holistic = mp.solutions.holistic

FACE_KEY_INDICES = [
    0, 17, 37, 39, 40, 61, 78, 80, 81, 82, 84, 87, 88, 91, 95,
    146, 178, 181, 185, 191, 267, 269, 270, 291, 308, 310, 311,
    312, 314, 317, 318, 321, 324, 375, 402, 405, 409, 415,
    46, 53, 52, 65, 55, 70, 276, 283, 282, 295, 285,
    33, 7, 163, 144, 145, 153, 154, 155, 133,
    362, 382, 381, 380, 374, 373, 390, 249, 263,
]

# Feature layout per frame:
# Pose:       33 × 3 = 99
# Face (key): 68 × 3 = 204
# Left hand:  21 × 3 = 63
# Right hand: 21 × 3 = 63
# TOTAL raw:           429
# + Hand velocity:     126  (63 lh + 63 rh)
# FINAL:               552

HAND_START = 99 + len(FACE_KEY_INDICES) * 3  # 303

ALL_CLASSES = [
    'we', 'plane', 'new', 'you', 'healthy', 'cool', 'brother', 'hello',
    'fast', 'he', 'hot', 'boy', 'sick', 'happy', 'good_morning', 'bad',
    'she', 'slow', 'train', 'i', 'thank_you', 'how_are_you', 'old', 'good', 'they'
    # no_gesture excluded intentionally
]

print(f"HAND_START        : {HAND_START}")
print(f"Raw features/frame: {99 + len(FACE_KEY_INDICES)*3 + 63 + 63}")
print(f"Final features/frame (with velocity): {99 + len(FACE_KEY_INDICES)*3 + 63 + 63 + 126}")
print(f"Classes: {len(ALL_CLASSES)}")

In [ ]:
def extract_landmarks(results):
    # Pose — anchor: nose (index 0)
    if results.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z] for lm in results.pose_landmarks.landmark])
        pose = (pose - pose[0]).flatten()
    else:
        pose = np.zeros(33 * 3)

    # Face (key landmarks only) — anchor: nose tip (index 1)
    if results.face_landmarks:
        all_face = np.array([[lm.x, lm.y, lm.z] for lm in results.face_landmarks.landmark])
        all_face = all_face - all_face[1]
        face = all_face[FACE_KEY_INDICES].flatten()
    else:
        face = np.zeros(len(FACE_KEY_INDICES) * 3)

    # Left hand — anchor: wrist (index 0)
    if results.left_hand_landmarks:
        lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark])
        lh = (lh - lh[0]).flatten()
    else:
        lh = np.zeros(21 * 3)

    # Right hand — anchor: wrist (index 0)
    if results.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark])
        rh = (rh - rh[0]).flatten()
    else:
        rh = np.zeros(21 * 3)

    return np.concatenate([pose, face, lh, rh])  # (429,)


def add_velocity(sequence):
    sequence = np.array(sequence)                      # (30, 429)
    hand     = sequence[:, HAND_START:]                # (30, 126)
    vel      = np.zeros_like(hand)
    vel[1:]  = hand[1:] - hand[:-1]
    return np.concatenate([sequence, vel], axis=1)     # (30, 552)


print("✅ Feature functions defined")
print(f"   extract_landmarks output : 429 features")
print(f"   add_velocity output      : 552 features")

In [ ]:
def process_videos(dataset_path, classes):
    holistic = mp_holistic.Holistic(
        static_image_mode=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        model_complexity=1
    )

    X, y = [], []

    for sign in classes:
        sign_path = os.path.join(dataset_path, sign)
        if not os.path.isdir(sign_path):
            print(f"⚠ Skipping {sign}: folder not found")
            continue

        videos = [f for f in os.listdir(sign_path)
                  if f.lower().endswith(('.mp4', '.avi', '.mov'))]
        print(f"Processing '{sign}': {len(videos)} videos")

        for vid in videos:
            cap = cv2.VideoCapture(os.path.join(sign_path, vid))
            frames = []

            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                frame  = cv2.resize(frame, (640, 480))
                image  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image  = np.ascontiguousarray(image, dtype=np.uint8)
                result = holistic.process(image)
                frames.append(extract_landmarks(result))

            cap.release()

            if len(frames) == 0:
                print(f"  ⚠ No landmarks in {vid}, skipping")
                continue

            # Temporal normalisation → exactly 30 frames
            if len(frames) >= SEQUENCE_LENGTH:
                idx = np.linspace(0, len(frames)-1, SEQUENCE_LENGTH, dtype=int)
                seq = [frames[i] for i in idx]
            else:
                pad = [frames[-1]] * (SEQUENCE_LENGTH - len(frames))
                seq = frames + pad

            seq = add_velocity(seq)   # (30, 552)
            X.append(seq)
            y.append(sign)

    holistic.close()
    return np.array(X, dtype=np.float32), np.array(y)


X_raw, y_raw = process_videos(DATASET_PATH, ALL_CLASSES)

np.save(f'{WORKING_DIR}/X_raw.npy', X_raw)
np.save(f'{WORKING_DIR}/y_raw.npy', y_raw)

print(f"\n✅ Preprocessing done")
print(f"   X shape : {X_raw.shape}")   # (N, 30, 552)
print(f"   y shape : {y_raw.shape}")
print(f"   Classes : {np.unique(y_raw)}")

In [ ]:
def augment_train(X, y, n_aug=8):
    X_out, y_out = [], []
    for seq, label in zip(X, y):
        X_out.append(seq)
        y_out.append(label)
        for _ in range(n_aug):
            s = seq.copy()
            # 1. Time warp
            factor  = np.random.uniform(0.75, 1.30)
            new_len = max(2, int(SEQUENCE_LENGTH * factor))
            idx     = np.linspace(0, SEQUENCE_LENGTH-1, new_len).astype(int).clip(0, SEQUENCE_LENGTH-1)
            s       = s[idx]
            idx2    = np.linspace(0, len(s)-1, SEQUENCE_LENGTH).astype(int)
            s       = s[idx2]
            # 2. Landmark jitter
            s += np.random.normal(0, 0.007, s.shape)
            # 3. Scale
            s *= np.random.uniform(0.85, 1.15)
            # 4. Temporal shift
            s = np.roll(s, np.random.randint(-4, 5), axis=0)
            X_out.append(s)
            y_out.append(label)
    return np.array(X_out, dtype=np.float32), np.array(y_out)

print("✅ augment_train() defined")

In [ ]:
# Load raw
X = np.load(f'{WORKING_DIR}/X_raw.npy', allow_pickle=True)
y = np.load(f'{WORKING_DIR}/y_raw.npy', allow_pickle=True)

print(f"Raw: {X.shape}  — features/frame must be 552: {X.shape[2] == 552} ✅")

# Encode
le = LabelEncoder()
y_enc = le.fit_transform(y)
num_classes = len(le.classes_)
np.save(f'{WORKING_DIR}/label_classes.npy', le.classes_)
print(f"Classes ({num_classes}): {le.classes_}")

# ── SPLIT FIRST on raw, clean data ───────────────────────────────
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f"\nRaw train : {X_train_raw.shape}")
print(f"Clean test: {X_test.shape}   ← never touched by augmentation")

# ── AUGMENT only the training split ──────────────────────────────
X_train, y_train = augment_train(X_train_raw, y_train_raw, n_aug=8)
print(f"\nAugmented train: {X_train.shape}")

# ── SCALE ────────────────────────────────────────────────────────
scaler = RobustScaler()
X_train_sc = scaler.fit_transform(
    X_train.reshape(-1, X_train.shape[-1])
).reshape(X_train.shape)

X_test_sc = scaler.transform(
    X_test.reshape(-1, X_test.shape[-1])
).reshape(X_test.shape)

joblib.dump(scaler, f'{WORKING_DIR}/isl_scaler.pkl')
print(f"\n✅ RobustScaler saved — n_features_in_: {scaler.n_features_in_}")

# Class weights
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(cw))

print(f"\n✅ Ready for training")
print(f"   Train : {X_train_sc.shape}")
print(f"   Test  : {X_test_sc.shape}")

In [ ]:
def build_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)

    x = Conv1D(64, 3, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Conv1D(64, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.3)(x)

    x = Bidirectional(LSTM(64, return_sequences=False))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.3)(x)

    x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    outputs = Dense(num_classes, activation='softmax')(x)
    return Model(inputs, outputs)


model = build_model(
    input_shape=(X_train_sc.shape[1], X_train_sc.shape[2]),
    num_classes=num_classes
)

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=20,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=7, min_lr=1e-6, verbose=1),
    ModelCheckpoint(f'{WORKING_DIR}/best_isl_model.keras',
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

history = model.fit(
    X_train_sc, y_train,
    validation_data=(X_test_sc, y_test),
    epochs=150,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

# Evaluate
loss, acc = model.evaluate(X_test_sc, y_test, verbose=0)
print(f"\n✅ Test Accuracy: {acc:.4f}")
print(f"✅ Test Loss    : {loss:.4f}")

y_pred = np.argmax(model.predict(X_test_sc), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epochs'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epochs'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/training_curves.png', dpi=100)
plt.show()

# Confusion matrix
plt.figure(figsize=(18, 16))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
    cmap=plt.cm.Blues, ax=plt.gca(), xticks_rotation=45
)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/confusion_matrix.png', dpi=100)
plt.show()